# 02 — Preprocessing Pipeline Walkthrough

This notebook executes the preprocessing script and analyzes the generated splits, weights, and features.

In [ ]:
import subprocess
result = subprocess.run(['python', '../src/preprocess.py'], capture_output=True, text=True, cwd="..")
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

In [ ]:
import pandas as pd

X_train = pd.read_csv('../data/processed/X_train.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

print("\n--- Data Types in X_train ---")
print(X_train.dtypes)

In [ ]:
import joblib
feature_list = joblib.load('../data/processed/feature_list.joblib')
print(f"Saved feature list ({len(feature_list)} features):")
print(feature_list)

In [ ]:
import matplotlib.pyplot as plt

y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

plt.figure(figsize=(8, 5))
plt.hist(y_train, bins=20, alpha=0.5, color='blue', label='Train log_HER')
plt.hist(y_test, bins=20, alpha=0.5, color='orange', label='Test log_HER')
plt.xlabel("log_HER")
plt.ylabel("Density")
plt.title("Overlay of Train vs Test log_HER distributions")
plt.legend()
plt.show()

In [ ]:
weights = pd.read_csv('../data/processed/sample_weights_train.csv', header=None).squeeze()
plt.figure(figsize=(8, 4))
plt.hist(weights, bins=15, color='teal', edgecolor='k', alpha=0.7)
plt.xlabel("Sample Weight")
plt.ylabel("Count")
plt.title("Training Sample Weights Distribution")
plt.show()

In [ ]:
import os
encoder_path = '../data/processed/target_encoder.joblib'
if os.path.exists(encoder_path):
    encoder = joblib.load(encoder_path)
    # Get target-encoded values for unique categories in host_material
    # Since target encoder learned mapping, let's print encoder attributes or structure
    print("Target Encoder fitted parameters:")
    print(encoder)
    if hasattr(encoder, "encodings_"):
        # For older versions of sklearn it may differ, let's print encoding details safely
        # target encoder stores learned encodings in encodings_
        # Let's visualize the means or categories
        try:
            categories = encoder.categories_[0]
            encodings = encoder.encodings_[0]
            map_df = pd.DataFrame({'host_material': categories, 'encoded_value': encodings})
            map_df_sorted = map_df.sort_values(by='encoded_value', ascending=False)
            
            plt.figure(figsize=(10, 5))
            map_df_sorted.head(10).plot.bar(x='host_material', y='encoded_value', color='mediumpurple', edgecolor='k')
            plt.title("Top 10 Encoded Values for host_material")
            plt.ylabel("Target Encoded Value (log scale)")
            plt.show()
        except Exception as e:
            print("Could not extract encoding details directly:", e)
else:
    print("Target Encoder file not found.")